# Independent acoustic developmental LM — 50-hour log-Mel pilot

This notebook trains the new **independent** causal acoustic path. It does not alter or load the existing IPA-feature Phase 1 checkpoints. Select a T4 GPU before running.

Default mode downloads the public official BabySLM Providence `audio.zip` without TalkBank credentials and selects only archive paths labelled `MOT` or `FAT`. The archive is audio-only: this identifies speaker role but does not prove each utterance's addressee or provide a child-age trajectory. TalkBank preparation and an upload-ZIP route remain optional fallbacks. See `docs/acoustic_logmel_phase1.md` for scope and limitations.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT}[acoustic]', 'pytest'], check=True)
# Editable installs can retain a stale package path in an already-running Colab kernel.
import importlib
SOURCE_DIR = PROJECT / 'src'
if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f'Repository source directory is missing: {SOURCE_DIR}')
sys.path.insert(0, str(SOURCE_DIR)) if str(SOURCE_DIR) not in sys.path else None
importlib.invalidate_caches()
loaded = sys.modules.get('devlm')
if loaded is not None and not str(pathlib.Path(getattr(loaded, '__file__', '')).resolve()).startswith(str(SOURCE_DIR.resolve())):
    for name in list(sys.modules):
        if name == 'devlm' or name.startswith('devlm.'):
            del sys.modules[name]
import devlm
os.chdir(PROJECT)
print('Project ready:', PROJECT)
print('Commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('devlm import:', pathlib.Path(devlm.__file__).resolve())
assert str(pathlib.Path(devlm.__file__).resolve()).startswith(str(SOURCE_DIR.resolve())), 'devlm did not resolve from this repository src/'

In [ ]:
# Optional code-level smoke tests; they use synthetic WAV only and do not train your data.
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_acoustic_phase1.py'], check=True)

## Prepare BabySLM Providence, TalkBank Providence, or upload audio

`SOURCE_MODE = 'babyslm_providence'` is the default public, credential-free route. It downloads a roughly 13-GB archive, then accepts only files in `*_MOT/` and `*_FAT/` directories. It has leakage-safe session grouping from filenames, but no verified child age or per-utterance addressee metadata.

`SOURCE_MODE = 'providence'` is the optional direct TalkBank route. It first reads the Colab Secrets `TALK_BANK_EMAIL` and `TALK_BANK_PASSWORD`; these are preferred because terminal-style `getpass` is unreliable in Colab. If either is unavailable, it shows a visible password widget and asks you to click Continue then rerun that cell. It downloads the official transcript ZIP, downloads only needed linked media, cuts timestamped MOT/FAT/other adult tiers to PCM WAV, and creates the strict manifest. `directed_to_child=true` means parent-child corpus context; CHAT does not encode the addressee of every individual tier.

`SOURCE_MODE = 'upload_zip'` uploads one ZIP containing exactly one `.tsv`/`.csv` manifest and its referenced `.wav` files. Required manifest columns: `audio_path`, `corpus_id`, `session_id`, `target_child_age_months`, `source_corpus`, `speaker_role`, `directed_to_child`; optional: `recording_order`, `split`.

In [ ]:
import shutil, zipfile, importlib, pathlib, sys
from google.colab import files

# Lightweight import self-heal: this cell may be rerun long after setup without a kernel restart.
PROJECT = globals().get('PROJECT', pathlib.Path('/content/developmental_checkpoints_word_recognition'))
SOURCE_DIR = pathlib.Path(PROJECT) / 'src'
if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f'Missing {SOURCE_DIR}. Rerun the Runtime setup cell before preparing audio.')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
loaded = sys.modules.get('devlm')
if loaded is not None and not str(pathlib.Path(getattr(loaded, '__file__', '')).resolve()).startswith(str(SOURCE_DIR.resolve())):
    for name in list(sys.modules):
        if name == 'devlm' or name.startswith('devlm.'):
            del sys.modules[name]
import devlm
from devlm.acoustic import providence as _providence_module
print('Acoustic code verified:', pathlib.Path(_providence_module.__file__).resolve(), flush=True)
SOURCE_MODE = 'babyslm_providence'  # alternatives: providence, upload_zip
if SOURCE_MODE not in {'babyslm_providence', 'providence', 'upload_zip'}:
    raise ValueError('SOURCE_MODE must be babyslm_providence, providence, or upload_zip')
INPUT_ROOT = pathlib.Path('/content/acoustic_cds_input')
if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)
INPUT_ROOT.mkdir(parents=True)
globals().pop('AUDIO_MANIFEST', None)
if SOURCE_MODE == 'babyslm_providence':
    from devlm.acoustic.babyslm import prepare_babyslm_providence
    print('Preparing public BabySLM Providence caregiver-code subset; no TalkBank login is used.', flush=True)
    AUDIO_MANIFEST = prepare_babyslm_providence(
        INPUT_ROOT, train_hours=50.0, validation_hours=0.5, validation_fraction=0.1, seed=20261005,
    )
elif SOURCE_MODE == 'providence':
    print('Stage 1/4: resolving runtime-only TalkBank credentials...', flush=True)
    talkbank_email = talkbank_password = None
    try:
        from google.colab import userdata
        talkbank_email = userdata.get('TALK_BANK_EMAIL')
        talkbank_password = userdata.get('TALK_BANK_PASSWORD')
        if talkbank_email and talkbank_password:
            print('Stage 1/4 complete: Colab Secrets found.', flush=True)
    except Exception:
        # Secret missing/denied: never include an exception value, which could expose account context.
        talkbank_email = talkbank_password = None
    if not (talkbank_email and talkbank_password):
        manual = globals().get('_TALKBANK_MANUAL_CREDENTIALS')
        if manual:
            talkbank_email, talkbank_password = manual
            print('Stage 1/4 complete: runtime-only widget credentials available.', flush=True)
        else:
            import ipywidgets as widgets
            from IPython.display import display
            email_box = widgets.Text(description='TalkBank email:', layout=widgets.Layout(width='420px'))
            password_box = widgets.Password(description='Password:', layout=widgets.Layout(width='420px'))
            continue_button = widgets.Button(description='Continue with runtime-only credentials', button_style='primary')
            credential_status = widgets.HTML('<b>Colab Secrets were unavailable.</b> Enter credentials here; the password is masked and is not written to disk.')
            def keep_credentials(_):
                if not email_box.value or not password_box.value:
                    credential_status.value = '<b>Please enter both email and password.</b>'
                    return
                globals()['_TALKBANK_MANUAL_CREDENTIALS'] = (email_box.value, password_box.value)
                password_box.value = ''
                credential_status.value = '<b>Credentials are held only in this runtime. Click Run on this cell once more to start Providence preparation.</b>'
            continue_button.on_click(keep_credentials)
            display(widgets.VBox([credential_status, email_box, password_box, continue_button]))
            print('Stage 1/4 paused: click Continue above, then rerun this cell. Do not run later cells yet.', flush=True)
    if talkbank_email and talkbank_password:
        from devlm.acoustic.providence import prepare_providence
        print('Stage 2/4: authenticating to TalkBank and preparing Providence audio...', flush=True)
        AUDIO_MANIFEST = prepare_providence(
            INPUT_ROOT, email=talkbank_email, password=talkbank_password,
            train_hours=50.0, validation_hours=0.5, validation_fraction=0.1, seed=20261005,
        )
        del talkbank_email, talkbank_password
        globals().pop('_TALKBANK_MANUAL_CREDENTIALS', None)
    else:
        raise RuntimeError('Providence preparation is paused for credentials. Click Continue in the widget above, then rerun this cell; do not run later cells yet.')
else:
    print('Select exactly one ZIP containing the WAV files plus one manifest TSV/CSV.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Expected exactly one ZIP upload; rerun this cell.')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if not archive_name.lower().endswith('.zip'):
        raise ValueError('Input must be a ZIP archive.')
    archive_path = INPUT_ROOT / pathlib.Path(archive_name).name
    archive_path.write_bytes(archive_bytes)
    with zipfile.ZipFile(archive_path) as archive:
        safe_root = INPUT_ROOT.resolve()
        for member in archive.infolist():
            destination = (INPUT_ROOT / member.filename).resolve()
            try:
                destination.relative_to(safe_root)
            except ValueError as exc:
                raise ValueError(f'Unsafe ZIP member path: {member.filename!r}') from exc
        archive.extractall(INPUT_ROOT)
    manifests = sorted([p for p in INPUT_ROOT.rglob('*') if p.suffix.lower() in {'.tsv', '.csv'}])
    if len(manifests) != 1:
        raise ValueError(f'Expected exactly one TSV/CSV manifest in ZIP; found {len(manifests)}: {manifests}')
    AUDIO_MANIFEST = manifests[0]
if 'AUDIO_MANIFEST' in globals():
    print('Stage 2/4 complete. Manifest:', AUDIO_MANIFEST)

In [ ]:
# Fail-fast data audit: validates adult/CDS metadata and reports available duration.
import wave, torch
from collections import Counter
from devlm.acoustic.data import load_audio_manifest, split_audio_sessions

items = load_audio_manifest(AUDIO_MANIFEST)
def wav_seconds(path):
    with wave.open(str(path), 'rb') as handle:
        return handle.getnframes() / handle.getframerate()
seconds = sum(wav_seconds(item.audio_path) for item in items)
train_items, validation_items = split_audio_sessions(items, validation_fraction=0.1, seed=20261005)
train_seconds = sum(wav_seconds(item.audio_path) for item in train_items)
print(f'Validated {len(items):,} adult/caregiver-to-child WAV recordings, {seconds / 3600:.3f} total raw hours.')
print(f'After the deterministic session split: {train_seconds / 3600:.3f} train hours, {sum(wav_seconds(item.audio_path) for item in validation_items) / 3600:.3f} validation hours.')
print('Source corpora:', dict(Counter(item.source_corpus for item in items)))
print('Speaker roles:', dict(Counter(item.speaker_role for item in items)))
if train_seconds / 3600 < 50:
    print('WARNING: fewer than 50 training-split hours are available. The run will train on all eligible training-split audio, not invent missing exposure.')
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json, torch

OUTPUT_DIR = pathlib.Path('/content/devlm_acoustic_logmel_50h_outputs')
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)  # known local Colab output path only; prevents stale checkpoints on rerun
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def toml_string(value):
    return json.dumps(str(value))

config_text = f'''[acoustic_phase1]
audio_manifest_path = {toml_string(AUDIO_MANIFEST)}
output_dir = {toml_string(OUTPUT_DIR)}
seed = 20261005
device = "cuda"
validation_fraction = 0.1
sample_rate = 16000
n_mels = 80
n_fft = 400
hop_length = 160
hidden_size = 128
num_layers = 1
dropout = 0.0
learning_rate = 0.001
gradient_clip_norm = 1.0
sequence_chunk_frames = 4096
future_horizons_frames = [3, 5, 10]
max_train_hours = 50.0
max_validation_hours = 0.5
normalization_max_hours = 1.0
target_checkpoint_count = 30
'''
COLAB_CONFIG = pathlib.Path('/content/acoustic_logmel_50h.toml')
COLAB_CONFIG.write_text(config_text, encoding='utf-8')
print(config_text)

In [ ]:
# Causal 30/50/100-ms future log-Mel prediction; live progress and each checkpoint's validation loss print here.
subprocess.run([sys.executable, '-u', '-m', 'devlm.acoustic.cli', '--config', str(COLAB_CONFIG)], check=True)

In [ ]:
checkpoints = sorted(OUTPUT_DIR.glob('acoustic_checkpoint_step_*.pt'))
if not checkpoints:
    raise FileNotFoundError('No acoustic checkpoints found; finish the training cell first.')
archive_path = shutil.make_archive('/content/devlm_acoustic_logmel_50h_outputs', 'zip', root_dir=OUTPUT_DIR)
print(f'Archived {len(checkpoints)} acoustic checkpoints to {archive_path}')
files.download(archive_path)